# 06 – Data for the Tableau story

Small, ready tables for each Tableau dashboard, made from the dbt table `mart_sessions`.
They are saved as CSV in `dashboard/data/` and packed into the Tableau workbook.

In [1]:
from pathlib import Path
import pandas as pd

DATA_FILE = Path("../data/processed/mart_sessions_all.parquet")  # saved by notebook 04
OUT_DIR = Path("../dashboard/data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

sessions = pd.read_parquet(DATA_FILE)
FULL_DATA_USERS = 12_899_779
print(sessions.shape)

(1000000, 20)


## 1. Funnel

In [2]:
n_users = len(sessions)
funnel = pd.DataFrame({
    "step_order": [1, 2, 3],
    "step": ["Visited the shop", "Added to cart", "Ordered"],
    "users": [n_users, sessions["has_cart"].sum(), sessions["has_order"].sum()],
})
funnel["share"] = funnel["users"] / n_users
funnel.to_csv(OUT_DIR / "funnel.csv", index=False)
funnel

,step_order,step,users,share
0,1,Visited the shop,1000000,1.000000
1,2,Added to cart,294974,0.294974
2,3,Ordered,126166,0.126166


## 2. Buyer types

In [3]:
names = {
    "browser": "Only browsed",
    "abandoner": "Left the cart",
    "quick_buyer": "Bought within 1 hour",
    "same_day_buyer": "Bought in 1–24 hours",
    "slow_buyer": "Bought after 1+ day",
    "late_cart": "Cart on Aug 27–28",
    "other": "Order without cart",
}
groups = {
    "browser": "Browsed", "abandoner": "Left the cart",
    "quick_buyer": "Bought", "same_day_buyer": "Bought", "slow_buyer": "Bought",
    "late_cart": "Left out", "other": "Left out",
}

buyer_types = sessions["buyer_type"].value_counts().reset_index()
buyer_types.columns = ["buyer_type", "users"]
buyer_types["name"] = buyer_types["buyer_type"].map(names)
buyer_types["group"] = buyer_types["buyer_type"].map(groups)
buyer_types["share"] = buyer_types["users"] / n_users
buyer_types.to_csv(OUT_DIR / "buyer_types.csv", index=False)
buyer_types

,buyer_type,users,name,group,share
0,browser,702220,Only browsed,Browsed,0.702220
1,abandoner,159946,Left the cart,Left the cart,0.159946
2,quick_buyer,81723,Bought within 1 hour,Bought,0.081723
3,slow_buyer,22992,Bought after 1+ day,Bought,0.022992
4,same_day_buyer,16263,Bought in 1–24 hours,Bought,0.016263
5,late_cart,11668,Cart on Aug 27–28,Left out,0.011668
6,other,5188,Order without cart,Left out,0.005188


## 3. Time from cart to order

In [4]:
buyers = sessions[sessions["buyer_type"].isin(["quick_buyer", "same_day_buyer", "slow_buyer"])]
buyers = buyers[~buyers["starts_with_cart_or_order"]]
hours = buyers["hours_cart_to_order"]

buckets = pd.DataFrame({
    "bucket_order": [1, 2, 3, 4, 5],
    "bucket": ["Under 10 min", "10–60 min", "1–24 hours", "1–7 days", "More than a week"],
    "buyers": [
        (hours < 1 / 6).sum(),
        ((hours >= 1 / 6) & (hours < 1)).sum(),
        ((hours >= 1) & (hours < 24)).sum(),
        ((hours >= 24) & (hours < 168)).sum(),
        (hours >= 168).sum(),
    ],
})
buckets["share"] = buckets["buyers"] / buckets["buyers"].sum()
buckets.to_csv(OUT_DIR / "time_to_order.csv", index=False)
buckets

,bucket_order,bucket,buyers,share
0,1,Under 10 min,52096,0.436322
1,2,10–60 min,28562,0.239217
2,3,1–24 hours,16087,0.134734
3,4,1–7 days,12784,0.107070
4,5,More than a week,9869,0.082656


## 4. Who to contact

In [5]:
cart_users = sessions[sessions["buyer_type"].isin(["abandoner", "quick_buyer", "same_day_buyer", "slow_buyer"])]
abandoners = cart_users[cart_users["buyer_type"] == "abandoner"]
slow_buyers = cart_users[cart_users["buyer_type"] == "slow_buyer"]

few_adds = abandoners[abandoners["n_carts"] < 3]
many_adds = abandoners[abandoners["n_carts"] >= 3]

target = pd.DataFrame({
    "group_order": [1, 2, 3],
    "group": ["Abandoners, 1–2 cart adds", "Abandoners, 3+ cart adds (target)", "Slow buyers"],
    "users": [len(few_adds), len(many_adds), len(slow_buyers)],
    "median_cart_adds": [few_adds["n_carts"].median(), many_adds["n_carts"].median(), slow_buyers["n_carts"].median()],
    "median_clicks": [few_adds["n_clicks"].median(), many_adds["n_clicks"].median(), slow_buyers["n_clicks"].median()],
})
target.to_csv(OUT_DIR / "target_group.csv", index=False)
target

,group_order,group,users,median_cart_adds,median_clicks
0,1,"Abandoners, 1–2 cart adds",103326,1.0,8.0
1,2,"Abandoners, 3+ cart adds (target)",56620,5.0,26.0
2,3,Slow buyers,22992,7.0,55.0


## 5. Orders by weekday and hour

In [6]:
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
orders = sessions[sessions["has_order"]].copy()
orders["order_hour"] = orders["order_hour"].astype(int)

heat = orders.groupby(["order_weekday", "order_hour"]).size().reset_index(name="orders")
heat["weekday_order"] = heat["order_weekday"].map({d: i + 1 for i, d in enumerate(days)})
heat.to_csv(OUT_DIR / "orders_by_hour.csv", index=False)
heat.head()

,order_weekday,order_hour,orders,weekday_order
0,Friday,0,303,5
1,Friday,1,158,5
2,Friday,2,82,5
3,Friday,3,55,5
4,Friday,4,38,5


## 6. Key numbers

In [ ]:
n_cart_users = len(cart_users)
n_abandoners = len(abandoners)
current_buyers = sessions["has_order"].sum()

kpis = pd.DataFrame([
    ["users", n_users],
    ["full_data_users", FULL_DATA_USERS],
    ["cart_rate", sessions["has_cart"].mean()],
    ["order_rate", sessions["has_order"].mean()],
    ["abandonment_rate", n_abandoners / n_cart_users],
    ["abandoners", n_abandoners],
    ["current_buyers", current_buyers],
    ["median_minutes_to_order", hours.median() * 60],
    ["target_group", len(many_adds)],
], columns=["metric", "value"])
kpis.to_csv(OUT_DIR / "kpis.csv", index=False)
kpis